In [1]:
# fetch pytorch model from .pt
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import classification_report, f1_score

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")


# ── Paths ────────────────────────────────────────────────────────────────────

ROOT = find_project_root() 
MODEL_PATH = ROOT / "data/mlData/model/Bravo_final.pt"
TEST_PATH  = ROOT / "data/mlData/trainData/202603-vX-test.jsonl"

SEQ_LEN    = 20
BATCH_SIZE = 512
LABEL_MAP  = {-1: 0, 0: 1, 1: 2}
CLASS_NAMES = ["DOWN", "NO_HIT", "UP"]

# ── Model architecture (mirrors lstmXII.py TripleBarrierLSTM) ─────────────
class TripleBarrierLSTM(nn.Module):
    def __init__(self, n_features, seq_len=20,
                 lstm1_hidden=128, lstm2_hidden=64, dense_hidden=32,
                 input_dropout=0.10, recurrent_dropout=0.20, dense_dropout=0.20, n_classes=3):
        super().__init__()
        self.input_dropout = nn.Dropout(p=input_dropout)
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=lstm1_hidden,
                            num_layers=2, batch_first=True,
                            dropout=recurrent_dropout, bidirectional=False)
        self.hidden_proj = nn.Linear(lstm1_hidden, lstm2_hidden)
        self.batch_norm  = nn.BatchNorm1d(lstm2_hidden)
        self.dense = nn.Sequential(
            nn.Linear(lstm2_hidden, dense_hidden),
            nn.ReLU(),
            nn.Dropout(p=dense_dropout),
            nn.Linear(dense_hidden, n_classes),
        )
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=True):
        x = self.input_dropout(x)
        _, (h_n, _) = self.lstm(x)
        ctx = h_n[-1]
        ctx = self.hidden_proj(ctx)
        ctx = self.batch_norm(ctx)
        logits = self.dense(ctx)
        return logits if return_logits else self.softmax(logits)

# ── Load checkpoint ───────────────────────────────────────────────────────
ckpt         = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
feature_cols = ckpt["feature_cols"]
mkw          = ckpt["model_kwargs"]
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TripleBarrierLSTM(n_features=mkw["n_features"], seq_len=mkw["seq_len"])
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()

print(f"Loaded  : {MODEL_PATH.name}")
print(f"Run     : {ckpt['run_name']}  |  best epoch: {ckpt['best_epoch']}")
print(f"Val     : loss={ckpt['val_loss']:.4f}  acc={ckpt['val_acc']:.4f}  F1={ckpt['val_f1']:.4f}")
print(f"Features: {len(feature_cols)}  →  {feature_cols}")
print(f"Device  : {device}")

# ── Load test data ────────────────────────────────────────────────────────
X_rows, y_rows = [], []
with open(TEST_PATH) as f:
    for line in f:
        row = json.loads(line)
        lbl = row.get("label")
        if lbl is None:
            continue
        X_rows.append([row[c] for c in feature_cols])
        y_rows.append(LABEL_MAP[int(lbl)])

X = np.array(X_rows, dtype=np.float32)
y = np.array(y_rows, dtype=np.int64)
print(f"\nTest    : {len(X):,} rows  →  {len(X) - SEQ_LEN:,} sequences")

# ── Sliding-window inference ──────────────────────────────────────────────
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.long)

all_probs, all_true = [], []
with torch.no_grad():
    for start in range(0, len(X_t) - SEQ_LEN, BATCH_SIZE):
        end     = min(start + BATCH_SIZE, len(X_t) - SEQ_LEN)
        seqs    = torch.stack([X_t[i:i + SEQ_LEN] for i in range(start, end)]).to(device)
        labels  = y_t[start + SEQ_LEN : end + SEQ_LEN]
        probs   = model(seqs, return_logits=False).cpu().numpy()
        all_probs.append(probs)
        all_true.extend(labels.tolist())

probs_arr  = np.vstack(all_probs)           # [N, 3]
y_pred_arr = probs_arr.argmax(axis=1)
y_true_arr = np.array(all_true)

# ── Classification report ─────────────────────────────────────────────────
test_f1  = f1_score(y_true_arr, y_pred_arr, average="macro", zero_division=0)
val_f1   = ckpt["val_f1"]
gap      = val_f1 - test_f1

print(f"\n{'='*60}")
print(f"  Val  F1 macro : {val_f1:.4f}")
print(f"  Test F1 macro : {test_f1:.4f}")
print(f"  Regime gap    : {gap:.4f}", end="  ")
if   gap < 0.02:  print("← generalises well")
elif gap < 0.08:  print("← expected structural gap (normal)")
elif gap < 0.15:  print("← WARN: moderate overfitting to bull regime")
else:             print("← ALARM: severe regime overfitting")
print(f"{'='*60}")
print(classification_report(y_true_arr, y_pred_arr, target_names=CLASS_NAMES, digits=4))


Loaded  : Bravo_final.pt
Run     : Run52_x  |  best epoch: 3
Val     : loss=1.0553  acc=0.4145  F1=0.3979
Features: 22  →  ['ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1', 'ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DELTA_1', 'DELTA_3', 'VOL_SPIKE', 'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS']
Device  : cpu

Test    : 54,589 rows  →  54,569 sequences

  Val  F1 macro : 0.3979
  Test F1 macro : 0.3935
  Regime gap    : 0.0044  ← generalises well
              precision    recall  f1-score   support

        DOWN     0.3898    0.5097    0.4418     18316
      NO_HIT     0.4630    0.5523    0.5037     18134
          UP     0.3546    0.1759    0.2352     18119

    accuracy                         0.4130     54569
   macro avg     0.4025    0.4126    0.3935     54569
weighted avg     0.4024    0.4130    0.3937     54569



In [2]:

# ── Conviction threshold sweep ────────────────────────────────────────────
# run on full test data window 
# probs_arr and y_true_arr from cell above: [N, 3]
p_dn     = probs_arr[:, 0]
p_no_hit = probs_arr[:, 1]
p_up     = probs_arr[:, 2]

conv_up = p_up - np.maximum(p_no_hit, p_dn)
conv_dn = p_dn - np.maximum(p_no_hit, p_up)

print(f"{'Threshold':>10}  {'N_LONG':>7}  {'N_SHORT':>8}  {'N_Sig':>7}  "
      f"{'SigRate%':>9}  {'Prec_UP':>8}  {'Prec_DN':>8}  {'F1mac_sig':>10}")
print("-" * 85)

for thr in np.arange(0.01,0.41,0.01):
    long_mask  = (conv_up > thr) & ~(conv_dn > thr)
    short_mask = (conv_dn > thr) & ~(conv_up > thr)
    sig_mask   = long_mask | short_mask

    n_long  = long_mask.sum()
    n_short = short_mask.sum()
    n_sig   = sig_mask.sum()
    rate    = 100.0 * n_sig / len(y_true_arr)

    if n_sig == 0:
        print(f"{thr:>10.2f}  {n_long:>7}  {n_short:>8}  {n_sig:>7}  {rate:>8.2f}%  {'—':>8}  {'—':>8}  {'—':>10}")
        continue

    # precision: among LONG signals, how many true label == UP (2)?
    prec_up = (y_true_arr[long_mask]  == 2).mean() if n_long  > 0 else float("nan") # NaN = no signals fired, precision is undefined
    prec_dn = (y_true_arr[short_mask] == 0).mean() if n_short > 0 else float("nan")

    # macro F1 on the filtered signal subset
    y_sub_true = y_true_arr[sig_mask]
    y_sub_pred = np.where(long_mask[sig_mask], 2, 0)   # LONG→UP(2), SHORT→DN(0)
    f1_sig = f1_score(
        y_sub_true, y_sub_pred,
        labels=[0, 2], average="macro", zero_division=0
    )

    print(f"{thr:>10.2f}  {n_long:>7}  {n_short:>8}  {n_sig:>7}  {rate:>8.2f}%  "
          f"{prec_up:>8.4f}  {prec_dn:>8.4f}  {f1_sig:>10.4f}")


 Threshold   N_LONG   N_SHORT    N_Sig   SigRate%   Prec_UP   Prec_DN   F1mac_sig
-------------------------------------------------------------------------------------
      0.01     4376     18116    22492     41.22%    0.3579    0.4027      0.3911
      0.02     1979     14144    16123     29.55%    0.3603    0.4136      0.3687
      0.03      624     10836    11460     21.00%    0.4071    0.4226      0.3412
      0.04      147      8180     8327     15.26%    0.4694    0.4319      0.3196
      0.05        4      6145     6149     11.27%    0.5000    0.4358      0.3042
      0.06        0      4576     4576      8.39%       nan    0.4471      0.3090
      0.07        0      3449     3449      6.32%       nan    0.4590      0.3146
      0.08        0      2529     2529      4.63%       nan    0.4678      0.3187
      0.09        0      1865     1865      3.42%       nan    0.4702      0.3198
      0.10        0      1420     1420      2.60%       nan    0.4775      0.3232
      0.11  

In [3]:
# check average probability distribution
# Test data window got weird result
# tried ru sweep per regime

print(f"Mean P(UP):   {p_up.mean():.4f}   Max: {p_up.max():.4f}")
print(f"Mean P(DN):   {p_dn.mean():.4f}   Max: {p_dn.max():.4f}")
print(f"Mean P(NONE): {p_no_hit.mean():.4f}   Max: {p_no_hit.max():.4f}")

# Distribution of conv_up
print(f"\nconv_up > 0.01: {(conv_up > 0.01).sum()}")
print(f"conv_up > 0.03: {(conv_up > 0.03).sum()}")
print(f"conv_up > 0.05: {(conv_up > 0.05).sum()}")
print(f"conv_up max:    {conv_up.max():.4f}")

# ── load timestamps from JSONL (aligned to sequence positions) ────────────
from datetime import datetime, timezone

with open(TEST_PATH) as f:
    rows_ts = [json.loads(l) for l in f if json.loads(l).get("label") is not None]

seq_ts = np.array([rows_ts[i + SEQ_LEN]["timestamp"] for i in range(len(probs_arr))])
fmt_ts = lambda ts: datetime.fromtimestamp(ts / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

# label distribution
long_mask_005 = conv_up > 0.05
long_indices  = np.where(long_mask_005)[0]
# signal at position i → bar at i + SEQ_LEN in original sequence
print(f"\nLong signal bar indices: {long_indices + SEQ_LEN}")
print(f"Long signal timestamps:")
for idx in long_indices:
    print(f"  seq[{idx:>5}]  bar[{idx + SEQ_LEN:>5}]  {fmt_ts(seq_ts[idx])}")

Mean P(UP):   0.3366   Max: 0.4688
Mean P(DN):   0.3596   Max: 0.5852
Mean P(NONE): 0.3038   Max: 0.7604

conv_up > 0.01: 4376
conv_up > 0.03: 624
conv_up > 0.05: 4
conv_up max:    0.0531

Long signal bar indices: [3148 3194 5043 5045]
Long signal timestamps:
  seq[ 3128]  bar[ 3148]  2025-09-13 09:20 UTC
  seq[ 3174]  bar[ 3194]  2025-09-13 13:10 UTC
  seq[ 5023]  bar[ 5043]  2025-09-19 23:15 UTC
  seq[ 5025]  bar[ 5045]  2025-09-19 23:25 UTC


In [4]:
# ── Threshold sweep per regime ────────────────────────────────────────────
# Requires: probs_arr, y_true_arr from cell 1  (no re-inference needed)
# Attaches regime label per sequence by reading 202603-vX-test-regime-mapped.jsonl

REGIME_PATH = ROOT / "data/mlData/trainData/202603-vX-test-regime-mapped.jsonl"
REGIMES     = ["ACCUMULATION", "BULLISH", "RECOVERY", "DISTRIBUTION", "BEARISH", "CORRECTION", "UNKNOWN"]

# ── Load regime per bar, aligned to same row order as TEST_PATH ───────────
regime_rows = []
with open(REGIME_PATH) as f:
    for line in f:
        row = json.loads(line)
        if row.get("label") is None:
            continue
        regime_rows.append(row["regime"])

# sequence i predicts at bar i+SEQ_LEN  →  use regime of that prediction bar
regime_arr = np.array([regime_rows[i + SEQ_LEN] for i in range(len(probs_arr))])

print(f"Total sequences : {len(regime_arr):,}")
for r in REGIMES:
    n = (regime_arr == r).sum()
    print(f"  {r:15}: {n:>6,}  ({100 * n / len(regime_arr):.1f}%)")


# ── Sweep function ────────────────────────────────────────────────────────
def regime_sweep(probs, y_true, label="ALL", thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.01, 0.21, 0.01)

    p_dn     = probs[:, 0]
    p_no_hit = probs[:, 1]
    p_up     = probs[:, 2]
    conv_up  = p_up - np.maximum(p_no_hit, p_dn)
    conv_dn  = p_dn - np.maximum(p_no_hit, p_up)

    print(f"\n{'='*90}")
    print(f"  REGIME: {label}  (n={len(y_true):,})")
    print(f"{'='*90}")
    print(f"{'Threshold':>10}  {'N_LONG':>7}  {'N_SHORT':>8}  {'N_Sig':>7}  "
          f"{'SigRate%':>9}  {'Prec_UP':>8}  {'Prec_DN':>8}  {'F1mac_sig':>10}")
    print("-" * 90)

    for thr in thresholds:
        long_mask  = (conv_up > thr) & ~(conv_dn > thr)
        short_mask = (conv_dn > thr) & ~(conv_up > thr)
        sig_mask   = long_mask | short_mask

        n_long  = long_mask.sum()
        n_short = short_mask.sum()
        n_sig   = sig_mask.sum()
        rate    = 100.0 * n_sig / len(y_true)

        if n_sig == 0:
            print(f"{thr:>10.2f}  {n_long:>7}  {n_short:>8}  {n_sig:>7}  {rate:>8.2f}%  "
                  f"{'—':>8}  {'—':>8}  {'—':>10}")
            continue

        prec_up = (y_true[long_mask]  == 2).mean() if n_long  > 0 else float("nan")
        prec_dn = (y_true[short_mask] == 0).mean() if n_short > 0 else float("nan")

        y_sub_true = y_true[sig_mask]
        y_sub_pred = np.where(long_mask[sig_mask], 2, 0)   # LONG→UP(2), SHORT→DN(0)
        f1_sig = f1_score(y_sub_true, y_sub_pred,
                          labels=[0, 2], average="macro", zero_division=0)

        pu_s = f"{prec_up:.4f}" if not np.isnan(prec_up) else "      —"
        pd_s = f"{prec_dn:.4f}" if not np.isnan(prec_dn) else "      —"
        print(f"{thr:>10.2f}  {n_long:>7}  {n_short:>8}  {n_sig:>7}  {rate:>8.2f}%  "
              f"{pu_s:>8}  {pd_s:>8}  {f1_sig:>10.4f}")


# ── Run full sweep per regime + overall reference ─────────────────────────
for r in REGIMES:
    mask = regime_arr == r
    regime_sweep(probs_arr[mask], y_true_arr[mask], label=r)

regime_sweep(probs_arr, y_true_arr, label="ALL (reference)")


# ── Cross-regime precision comparison at key thresholds ──────────────────
KEY_THRESHOLDS = [0.03, 0.05, 0.07, 0.10, 0.13]

print(f"\n{'='*70}")
print(f"  CROSS-REGIME PRECISION SUMMARY")
print(f"{'='*70}")
print(f"{'Regime':>15}  {'thr':>5}  {'N_LONG':>7}  {'N_SHORT':>8}  {'Prec_UP':>8}  {'Prec_DN':>8}")
print("-" * 70)

for r in REGIMES:
    mask = regime_arr == r
    p    = probs_arr[mask]
    yt   = y_true_arr[mask]

    p_dn_r     = p[:, 0]
    p_no_hit_r = p[:, 1]
    p_up_r     = p[:, 2]
    c_up = p_up_r - np.maximum(p_no_hit_r, p_dn_r)
    c_dn = p_dn_r - np.maximum(p_no_hit_r, p_up_r)

    for thr in KEY_THRESHOLDS:
        lm  = (c_up > thr) & ~(c_dn > thr)
        sm  = (c_dn > thr) & ~(c_up > thr)
        pu  = (yt[lm] == 2).mean() if lm.sum() > 0 else float("nan")
        pd  = (yt[sm] == 0).mean() if sm.sum() > 0 else float("nan")
        pu_s = f"{pu:.4f}" if not np.isnan(pu) else "      —"
        pd_s = f"{pd:.4f}" if not np.isnan(pd) else "      —"
        print(f"{r:>15}  {thr:>5.2f}  {lm.sum():>7}  {sm.sum():>8}  {pu_s:>8}  {pd_s:>8}")
    print()

Total sequences : 54,569
  ACCUMULATION   : 13,207  (24.2%)
  BULLISH        :  8,337  (15.3%)
  RECOVERY       :  1,029  (1.9%)
  DISTRIBUTION   :  7,192  (13.2%)
  BEARISH        : 21,567  (39.5%)
  CORRECTION     :  3,237  (5.9%)
  UNKNOWN        :      0  (0.0%)

  REGIME: ACCUMULATION  (n=13,207)
 Threshold   N_LONG   N_SHORT    N_Sig   SigRate%   Prec_UP   Prec_DN   F1mac_sig
------------------------------------------------------------------------------------------
      0.01     1720      4313     6033     45.68%    0.3529    0.4062      0.4181
      0.02      788      3150     3938     29.82%    0.3693    0.4168      0.4069
      0.03      237      2243     2480     18.78%    0.4557    0.4213      0.3795
      0.04       63      1558     1621     12.27%    0.4921    0.4454      0.3509
      0.05        4      1124     1128      8.54%    0.5000    0.4511      0.3151
      0.06        0       782      782      5.92%         —    0.4629      0.3164
      0.07        0       568   

/var/folders/_x/v502z23d7mb3hlbnw0sdg6hh0000gn/T/ipykernel_20006/1108118210.py:52: RuntimeWarning: invalid value encountered in scalar divide
  rate    = 100.0 * n_sig / len(y_true)


# Threshold to execution

In [6]:
# Problem - test period didn't cover all regime
# Test : why bullish signal collapse in bullish regime

from datetime import datetime, timezone

# regime_labels: reuse regime_arr computed in the regime-sweep cell
regime_labels = regime_arr   # np.array of strings, length == len(probs_arr)

# test_timestamps: datetime objects aligned to sequences (seq_ts is ms epoch from cell b6dbcf35)
test_timestamps = np.array([
    datetime.fromtimestamp(ts / 1000, tz=timezone.utc)
    for ts in seq_ts
])

for regime in ['ACCUMULATION', 'BULLISH']:
    mask = (regime_labels == regime)
    conv_up_regime = conv_up[mask]
    
    print(f"\n{regime}")
    print(f"  conv_up max:      {conv_up_regime.max():.4f}")
    print(f"  conv_up mean:     {conv_up_regime.mean():.4f}")
    print(f"  conv_up > 0.01:   {(conv_up_regime > 0.01).sum()}")
    print(f"  conv_up > 0.03:   {(conv_up_regime > 0.03).sum()}")
    print(f"  conv_up > 0.05:   {(conv_up_regime > 0.05).sum()}")
    
    # When do they appear
    regime_indices = np.where(mask)[0]
    long_within = regime_indices[conv_up_regime > 0.01]
    if len(long_within) > 0:
        ts = test_timestamps[long_within]
        print(f"  First long: {ts.min()}")
        print(f"  Last long:  {ts.max()}")
        print(f"  Spread:     {(ts.max() - ts.min()).days} days")


ACCUMULATION
  conv_up max:      0.0531
  conv_up mean:     -0.0492
  conv_up > 0.01:   1720
  conv_up > 0.03:   237
  conv_up > 0.05:   4
  First long: 2025-09-03 18:30:00+00:00
  Last long:  2026-03-10 21:40:00+00:00
  Spread:     188 days

BULLISH
  conv_up max:      0.0430
  conv_up mean:     -0.1058
  conv_up > 0.01:   223
  conv_up > 0.03:   15
  conv_up > 0.05:   0
  First long: 2025-09-03 18:25:00+00:00
  Last long:  2026-03-10 12:35:00+00:00
  Spread:     187 days


In [8]:
# bulllish test
bullish_mask   = (regime_labels == 'BULLISH')
bullish_pup    = p_up[bullish_mask]
bullish_pdn    = p_dn[bullish_mask]

mid = len(bullish_pup) // 2
print(f"BULLISH first half  mean P(UP): {bullish_pup[:mid].mean():.4f}")
print(f"BULLISH second half mean P(UP): {bullish_pup[mid:].mean():.4f}")
print(f"BULLISH first half  mean P(DN): {bullish_pdn[:mid].mean():.4f}")
print(f"BULLISH second half mean P(DN): {bullish_pdn[mid:].mean():.4f}")

up_dominant = (bullish_pup > bullish_pdn).mean()
print(f"\nBULLISH bars where P(UP) > P(DN): {up_dominant:.1%}")

# Same for ACCUMULATION
accum_mask = (regime_labels == 'ACCUMULATION')
accum_pup  = p_up[accum_mask]
accum_pdn  = p_dn[accum_mask]
up_dom_acc = (accum_pup > accum_pdn).mean()
print(f"ACCUMULATION bars where P(UP) > P(DN): {up_dom_acc:.1%}")

# Global baseline
print(f"\nALL bars where P(UP) > P(DN): {(p_up > p_dn).mean():.1%}")

BULLISH first half  mean P(UP): 0.3277
BULLISH second half mean P(UP): 0.3192
BULLISH first half  mean P(DN): 0.3639
BULLISH second half mean P(DN): 0.3522

BULLISH bars where P(UP) > P(DN): 7.6%
ACCUMULATION bars where P(UP) > P(DN): 27.5%

ALL bars where P(UP) > P(DN): 17.7%
